# 8. Error Handling

## 8.0x. Comprehension & Retention Strategy 

### 8.0x.0. Overview

For this Chapter to experiment with comprehension and retention of the material, I will explore covering the chapter with some guided prompts. Breaking down the prompts into
- Before Reading
- During
- After

The idea is to break down a reading block into per sub-section and further into paragraphs is the information is too dense.

Question is whether it would be appropriate to skim through the whole chapter first before doing a second thorough read. I'm finding myself in that approach. Then the next chapter (Chapter 9) we can see if skimming the materail first has a benefit.

Thinking whether it would also be good to state assumptions about the chapter before skimming and then be able to validate if the assummptions were true.

Also thinking it would be good to take note of questions raised during the skimming and parts that I think need special/close attention to once we do a second read.

The prompts are inspired by some questions/prompts suggested by different LLMs. Refer to links below
- [ChatGPT](https://chatgpt.com/c/6a0d72f9-66b8-83ea-b983-c2eb4f529cc8)
- [Gemini Pro](https://gemini.google.com/app/7523f072b0032685)
- [Claude Opus 4.6](https://claude.ai/share/58fe2e3f-0294-4848-90f4-e70021b183c4)

### 8.0x.1. Before Reading

#### 8.0x.1.0. Intro

Here are some good prompts that would be good for the _Before Reading Phase_.

- _Why do I think this Chapter important?_
- _What do I think I already know about this Chapter?_
- _What do I think I will learn in this Chapter?_

I think those questions suffice for now. Might update later.

#### 8.0x.0.1. Why Do I Think This Chapter Important?

I think **Error Handling** is an important chapter to get into because
- We get to think about invalid states that an application might be in preemptively. Thss makes our app
  less buggy and more reliable because we address potential error paths up-front instead of waiting to
  be supprised later in production.
- Makes it easier for a developer to debug the an application faster because a lot of the data and
  context has been already been baked into a error when it thrown.

#### 8.0x.0.2. What Do I Think I Already Know About This Chapter?

Not too much, to be honest. Coming from a python background where use use raise and throw `exceptions` and I think rust prefers returning `errors` and allowing callers to decide whether to crash the program or return the error as a value. Have watched some videos on exceptions vs error values, but not sure i've completely grokked the difference.<br/>
One thing I will say if I recall what _The Book_ says about this topic is that there are basically 2 
types of errors
1. Recoverable Errors
2. Unrecoverable Errors

And you want to use different approaches when it comes to each. And it is important to make sure we handle all fallible operations appropriately and not just call `unwrap()` on them.

#### 8.0x.0.2. What Do I Think I Will Learn From This Chapter?

- Right now the major thing that comes to mind is that, so far we are just returning `StatusCode::INTERNAL_SERVER_ERROR` for the situation where either a `sqlx` operation or `reqwest`
operation fails. We probably want to handle this better so that we return more descriptive errors incase of either of the failures. <br/>
- We might also learn how to recover from recoverables failures in our application.

### 8.0x.2. During Reading

During skimming and when we re-read more deeply these are some guiding questions i'm thinking.

_Skimming Phase_<br/>
- _What are some concepts/ideas that standout and why?_
    - Is it because the idea/concept feels interesting or challengeing
    - Why is it interesting or challenging to you?

_Deep Dive Phase_
- _Can I **summarize** and **explain** what I've read in my own words without looking?_
   - Internalize key definitions/ code snippets that I want to memorize and know by
     heart.
    - Summarize in a sentence or less
    - How would I explain this concept to a beginner
- _How does this connect to or challenge, something I already understand?_
    - What are some of my assumptions that were true vs were not. 
- _Am I still tracking with the author, or did my mind just wander?_
    - Where did my mind go and what is most distracting? 

### 8.0x.3. After Reading

Some question in this phase are
- _What was the main idea/mechanism/argument and What were the 3-5 most important
  ideas/supporting details?_
- _Where does my understanding break down, what were the more challenging concepts and
  ideas to wrap my head around & how to I get better?_
- _Where and how would I actually use this?_

### 8.0x.4. Summary

These are a lot of guiding/prompts but hopefully we'll have refined a lot of them by the time we are done with the chapter

## 8.0. Overview

### 8.0.0. Skimming: What did you notice and why? Any Questions

_**What?**_<br/>
I like the questions the author rises right of the bat;
1. How do errors fit within the broader architecture of our application?
2. **What does a _good_ error look like?**
3. **Who are errors for?**
4. Should we use libraries for this and which ones?

_**Why?**_<br/>
I thinks these are good questions because that means by the end of the chapter one a key question like;
> _What does good error handling look like for all interested/affected parties?_

Would be good to revisit this questions and see if it was answered satisfactorily. Like out of 10 how satisfied were you
with the answer? <br/>
1 - Not satisfied at all, 10 Very very satisfied.


_**Questions?**_<br/>
None

_**Reflections & Answers**_ <br/>
After skimming through the book, how satisfied am I with how the book answered this question:
> _What does good error hanlding look lie for all interested parties?_

Honestly I would say around a 6.5 - 7. I feel like a got a general oveview and lay of the land. Might have to suppliment with  
extra material to feel satisfied and confident about what good error handling should look like upto the tracling layer.

I feel like I would have wanted a bit more on how errors and tracing are connected.

## 8.1. What Is The Purpose Of Errors?

### 8.1.0 Overview

##### 8.1.1.1.0. Deep Dive: Summarize, ELI5, Connect

_**Summarize**_  
Here we walk through 2 examples that help demonstrate the core idea around error handing for the main stakeholders
- Operators - These are the developers implementating a library or writing an application
- Users - These are the users of your application or library


Example 1 uses the `store_token` method in `src/routes/subscriptions.rs` that we can in `subscribe`.  
It was implemented as follows;
```Rust
//! src/routes/subscriptions.rs
// [...]

#[tracing::instrument([...])]
pub async fn subscribe([...]) -> HttpResponse {
    // [...]
    if store_token(&mut transaction, &subscription_token, subscriber_id)
        .await
        .is_err 
    {
        return HttpResponse::InternalServerError().finish();
    }
    // [...]
}


#[tracing::instrument([...])]
async fn store_token(
    transaction: &mut Transaction<'_, Postgres>,
    subscription_token: &str,
    subscriber_id: Uuid,
) -> Result<(), sqlx::Error> {
    let query = sqlx::query!(
        r#"
            INSERT INTO subscription_tokens (subscription_token, subscriber_id)
            VALUES ($1, $2)
        "#,
        subscription_token,
        subscriber_id,
    );

    // Fallibale operation
    transaction.execute(query)
        .await
        .map_err(|e| {
            tracing::error!("Failed while executing store token query: {}", e);
            e
        })?;
    Ok(())
}
```

In example 2 we use the form validation `match form.0.try_into()` snippet where we are validating user input (username and email) ensuring  
they were parsed appropriately into our custom types otherwise we raise a `BadRequest` status code.
```Rust
//! src/routes/subscriptions.rs
// [...]

impl TryFrom<FormData> for NewSubscriber {
    type Error = String;
    fn try_from(val: FormData) -> Result<Self, Self::Error> {
        let username = SubscriberUsername::parse(val.username)?;
        let email = SubscriberEmail::parse(val.email)?;
        Ok(Self { username, email })
    }
}

#[tracing::instrument([...])]
pub async fn subscribe(
    form: web::Form<FormData>,
    db_pool: web::Data<PgPool>,
    email_client: web::Data<EmailClient>,
    base_url: web::Data<ApplicationBaseUrl>,
) -> HttpResponse {
    let new_subscriber = match form.0.try_into() {
        Ok(form) => form,
        Err(_) =>  return HttpResponse::BadRequest().finish(),
    };

    // [...]
}
```

One is definitely an internal error given by the fact that we are actually returning an `InternalServerError` response.  
Meaning this one needs to be handled internally by the engineering team.

The other is a validation error that is client/user facing we return a `BadRequest` response. This one a user/client can
fix by correcting their inputs to the application.

### 8.1.1. Internal Errors

#### 8.1.1.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>
I like the questions the author rises right of the bat;
1. How do errors fit within the broader architecture of our application?
2. **What does a _good_ error look like?**
3. **Who are errors for?**
4. Should we use libraries for this and which ones?

_**Why?**_<br/>
I thinks these are good questions because that means by the end of the chapter one a key question like;
> _What does good error handling look like for all interested/affected parties?_

Would be good to revisit this quiestion and see if it was answered satisfactorily. Like out of 10 how satisfied were you
with the answer? <br/>
1 - Not satisfied at all, 10 Very very satisfied.


_**Questions?**_<br/>
None

#### 8.1.1.1. Enable The Caller To React

##### 8.1.1.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>
The `sqlx::Error` enum.

_**Why?**_<br/>
Someone took the time to map out possible error states when executing an sqlx query. Key take away is the attention to detail and meticulousness it must 
have took to map this


_**Questions?**_<br/>
- Curious about the distribution of errors across the variants of the `sqlx::Error` enums. Are there some errors that are more common that others?
- Wonder if there's any telemetry that is collected within the rust language itself that informs what the compiler optimizes? Sure this is a rabbit hole

##### 8.1.1.1.0. Deep Dive: Summarize, ELI5, Connect

_**Summary.**_  
- Because operations can fail in _multiple different ways_ we might want to respond differently depending on what happened.

_**ELI 5.**_

Looking at the `store_token` operation more closely, if it transaction execution fails there might be a couple of reasons
- Network failure i.e. took to long to execute query
- The query had an issue in that maybe we violated some DB constraint i.e. the `subscription_token` wasn't unique

Looking at the skeleton of `sqlx::Error`, the error type for `execute`, we see some possible errors that might occur

```Rust
//! sqlx-core/src/errors.rs

pub enum Error {
    Configuration(/* */),
    Database(/* */),
    Io(/* */),
    Tls(/* */),
    Protocol(/* */),
    RowNotFound,
    TypeNotFound {/* */},
    ColumnIndexOutOfBound {/* */},
    ColumnNotFound(/* */),
    ColumnDecode {/* */},
    Decode(/* */),
    PoolTimedOut,
    PoolClosed,
    WorkerCrashed,
    Migrate(/* */),
}

```
`sqlx::Error` is implemented as an enum to allow operators(users) to match returned error and behave _differently_ depending on the  
underlying failue modes. For example we might want to return a `PoolTimedOut` while we probably want to give up on a `ColumnNotFound`

Right now we don't have the appropriate mechanism to branch based on the error scenario.



#### 8.1.1.1. Help An Operator To Trouble Shoot.

##### 8.1.1.1.0. Deep Dive: Summarize, ELI5, Connect

_**Summarize**_  
We need enough information and context about errors to be able to troubleshoot efficiently and effectively when they occur.

_**ELI 5**_
Currently we are doing a `tracing::error!` log in `store_token` that helps report a failure. We can then inspect the logs when  
investigation an issue with the `transaction`
```Rust
#[tracing::instrument([...])]
async fn store_token([...]) -> Result<(), sqlx::Error> {
    // [...]
    transaction.execute(query)
        .await
        .map_err(|e| {
            tracing::error!("Failed to execute store token query: {:?}", e);
            e
        })?;
    // [...]
}
```

_**Connect**_  
For `sqlx::Error` we are both a user/client and operator/implementor.

As a user/client `sqlx::Error` provides us with variants we can use to control how the application responds. And the errors returned just give use  
enough information to correct our implementation or retry.

As operators/implementators we can then write custom types, methods, or wrappers that extend what sqlx provides for ourselves as operators or the end users.

### 8.1.2. Errors At The Edge.

#### 8.1.2.1 Help A User To Troubleshoot

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>
**Transient** _meaning_: Lasting only for a short time, impermanent, or temporary.<br/>
So _transient errors_ mean errors that last only for a short while and can be retried.

_**Why?**_<br/>
Have seen the word transient a couple of times in a programming context. Now I know it means short lived.

Note the difference between _transient_ and _transitive_

_**Questions?**_<br/>
None

##### 8.1.1.1.0. Deep Dive: Summarize, ELI5, Connect

_**Summarize**_  
Users/Clients of our APIs need a _signal_ when a failure mode is encountered.  
We need to provide the user with the right and approriate ammount of information required for them to know if 
1. This is an issue they need to resolve on their end
2. This is an issue being resolved on the operator side  

Without leaking implemeantion details

_**ELI 5**_  

A user facing the `InternalServerError` raised by the `store_token` at most only needs to be informed to come back later when the service back up again  
without leaking what is in the realm of the operator. We do this currently by simply returning the `HttpResponse::InternalServerError.finish()`.
```Rust
//! src/routes/subscriptions.rs
// [...]

pub async fn subscribe(/* */) -> HttpResponse {
    // [...]
    if store_token(&mut transaction, &subscription_token, subscriber_id)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish()   
    }
}
```

A user facing the `BadRequest` error raised by validation of the form input `FormData` would probably want to know what they need to fix on their end  
for their input data to be valid. Currently though, we are just returning a `BadRequst` status code that is not as informative.
```Rust

//! src/routes/subscriptions.rs
// [...]

pub async fn subscribe(/* */) -> HttpResponse {
    // [...]
    let new_subsciber = match form.0.try_into() {
        Ok(form) => form,
        Error(_) => return HttpResponse::BadRequest().finish()
    };
}
```
Internally though we are logging via `tracing::error!` in the relavant domains of the email and username validation. Below is the `src/domain/subscription_email.rs`
```Rust
//! src/domain/subscription_email.rs
// [...]

impl SubscriptionEmail {
    pub fn parse(s: String) -> Result<Self, String> {
        if s.validate_email() {
            Ok(Self(s))
        } else {
            Err(format!("{} is not a valid subscriber email.", s))
        }
    }
}
```

Good for the operator (us), not so good for the user. To fix this we need to populate the response body associated with the `BadRequest()` status we return for the user
to know what to do.

_**Connect**_  

### 8.1.3. Summary

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>

Rust conf talk by Jane Lusby [Error Handling Isn't All About Errors]().

Liked the error handling matrix

$$
\begin{array}{c|c|c}
\hline
    & \textsf{Internal} & \textsf{At the edge} \\
\hline
\textsf{Control Flow} & \textsf{Types} & \textsf{Status Code } \\
\textsf{Reporting} & \textsf{Logs} & \textsf{Response Body}\\
\hline
\end{array}
$$


_**Why?**_<br/>
The author says we should listen to the talk immediately. Seems like it's super talk.

Biggest takeaway from the talk is, that there is a divide between
- library errors that should be very descriptive informing where things failed and why ( use `thiserror`).
- application errors that are communicated while application is running? ( use `anyhow`).

 Still not super clear. Worth revisiting.

_**Questions?**_<br/>
None

##### 8.1.1.1.0. Deep Dive: Summarize, ELI5, Connect

_**Summarize**_  
In understanding the best way to handle errors we need to understand;
1. Intent the purpose
    - Control Flow and/or
    - Reporting
2. Audience
    - Internal Operators (working on the implemetation) and/or
    - Users - users of the implementation
3. Techniques available
    - Types, methods, fields
    - Logs/Traces
    - StatusCodes
    - Response Bodies

This is where the matrix the author provides gives quite a good summary of how to think about errors on a high level.  
I use different semantics for my own internalization.  
$$
\begin{array}{c|cc}
\textsf{Purpose}               & \textsf{Audience}                  &                \\
\hline
                               & \textsf{Operators/Implementators}  & \textsf{Users} \\
\hline
\textsf{Control Flow} & \textsf{Types, methods, fields}             & \textsf{StatusCodes} \\
\textsf{Reporting}    & \textsf{Logs/Tracing}                       & \textsf{ResponseBody}\\
\hline
\end{array}
$$

_**ELI 5 (Explain Like I'm 5)**_  

_**Connect**_  


## 8.2. Error Reporting For Operators

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>

- Reflecting on the difference between a _generic_ type and a _trail object_

Generics have to be filled in at comptime, Trait objects can only be known at runtime.

Best example I can think of is when an event calltime. 

We can know that we need a specified call time when team members or vendors need to get the office
or arrive at an event venue. We call this variable `call_time` and we'll give it a concrete time X.pm/am.

However the actuall time that team members or vendors will arrive is dynamic only to be know when they arrive.

Another good example is lets say we have a singing competition that is only elidgible to people over 16 for example.
The singing competition is generic over the competitants as long as they are over 16 years old. This will be known
by sign-up because they need to fill in some key details. However wether they can sing or not will only be know
when they open there mouth to sing in person. So;
- Contestants is _generic_ can be known with reasonble certainty at sign-up (comptime)
- Ability to sing is a _trait_ - can only be assessed at time of singing (runtime - dynamic dispatch)


_**Why?**_<br/>

Reasoning about why we need _dynamic dispatch_ of Errors as _trait objects_.

_**Questions?**_<br/>

- The books mentions that there's meant to be a `chain` method on `Error`. Curious what `Error` they mean here. Is it
  `std::error::Error` or `actix_web::error::Error`?

### 8.2.0. Overview

### 8.2.1. Keeping Track Of The Error Root Cause

### 8.2.2. The Error Trait

## 8.3. Errors For Control Flow

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>

- Implementing the `From` `Debug` and `Display` traits manually for our error enums.
- Seeing how `thiserror` makes this easier for us using procedural macros

_**Why?**_<br/>
- Helpful to see the internals of how `std` traits are implemented for our custom types

_**Questions?**_<br/>



### 8.3.1. Layering

### 8.3.2. Modelling Errors As Enums

### 8.3.3. The Error Type Is Not Enough

### 8.3.4. Removing The Boilerplate With `thiserror`

## 8.4. Avoid "Ball Of Mud" Error Enums

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>

- We get to see why and where we would use the `anyhow` trait.


_**Why?**_<br/>

- Clears a misconception I had earlier by stating that `thiserror` for libraries and `anyhow` for
  applictions is  not the right framing, as we need to reason about intent.

  If the intent is to be able to respond differently to different errors and give the caller maximum flexibilty to be able to deal/respond with
  different errors accordingly and programmatically we should use `thiserror`

  If instead we just want the user to be informed of something that failed without needed an operator or user to respond programmatically use `anyhow`.
  Makes the crate names stand out a bit more
  - "_Deal with_ `thiserror`"
  - "`anyhow` _this error occurred_"

   <br/>

  For example i'm thinking a _retriable_ error is a good candidate for `thiserror` because we can deal with it programmatically. However something
  like a database failure there is no way to deal with it programmatically we have to get into the code and debug what went wrong and fix it. This is
  where `anyhow` comes in?

_**Questions?**_<br/>



### 8.4.0. Overview

### 8.4.1. Using `anyhow` As An Opaque Error Type

### 8.4.2. `anyhow` Or `thiserror`?

## 8.5. Who Should Log Errors?

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>

- The author says a good rule of thumb is to log errors where they are handle. Indicating that if we are using the try operator `?`
  for propagating errors to callers, we should not be logging.


_**Why?**_<br/>

- Curious how the callers should log it? Should we, at the call sight then use `.map_err(|e| {tracing::error!("...{}", e); e})` and
  remove it from the propagator?

_**Questions?**_<br/>
The ones stated above.


## 8.6. Summary

##### 8.1.2.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_<br/>

- Implementing error handling on the `confirm` handler is left as an exercise to the reader. 

_**Why?**_<br/>

- Like this

_**Questions?**_<br/>
None